# STEP 3: clean data

In [2]:
## 1. setup
import os
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print("Working in:", Path.cwd())

con = duckdb.connect("kkbox.duckdb")
con.sql("SET memory_limit='6GB'")          # about half your RAM is a safe choice
con.sql("SET temp_directory='data/tmp'")   # where DuckDB spills if memory runs out

def run_sql_file(path):
    """Run every statement in a .sql file."""
    for stmt in Path(path).read_text(encoding="utf-8").split(";"):
        if stmt.strip():
            con.sql(stmt)
    print("Done:", path)

Working in: d:\0statistic project\subscription-lifecycle-churn-analysis


In [ ]:
## 2. check 
### 2a Are 0-day plans cancellations?
con.sql("""
    SELECT payment_plan_days = 0 AS zero_day_plan, is_cancel, COUNT(*) AS n
    FROM raw_tx GROUP BY ALL ORDER BY ALL
""").df()



,zero_day_plan,is_cancel,n
0,False,0,21232635
1,False,1,873778
2,True,0,854136
3,True,1,18206


In [ ]:
### 2b Impossibly dates: expiry before the transaction
con.sql("""
    SELECT is_cancel,
           COUNT(*) FILTER (WHERE membership_expire_date < transaction_date) AS expire_before_tx,
           MIN(membership_expire_date) AS min_expire,
           MAX(membership_expire_date) AS max_expire
    FROM raw_tx GROUP BY 1
""").df()


,is_cancel,expire_before_tx,min_expire,max_expire
0,0,6462,19700101,20361015
1,1,152304,19700101,20221201


In [ ]:
### 2c  Exact duplicate transactions
con.sql("""
    SELECT (SELECT COUNT(*) FROM raw_tx)
         - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM raw_tx)) AS duplicate_rows
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_rows
0,3339


In [ ]:
### 2d Is each member listed once? What does age look like?
con.sql("""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT msno) AS n_users,
           COUNT(*) FILTER (WHERE bd BETWEEN 1 AND 9) AS age_1_to_9,
           COUNT(*) FILTER (WHERE bd BETWEEN 10 AND 90) AS age_plausible
    FROM raw_members
""").df()

,n_rows,n_users,age_1_to_9,age_plausible
0,6769473,6769473,899,2220365


In [ ]:
## 3. clean.sql: clean transactions, members and user logs
run_sql_file("sql/02_clean.sql")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done: sql/02_clean.sql


In [11]:
## 4. Build the cleaning log
log_queries = {
    ("transactions", "Exact duplicate rows removed"):
        "SELECT (SELECT COUNT(*) FROM raw_tx) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM raw_tx))",
    ("transactions", "Payments with expiry before payment date dropped"):
        "SELECT COUNT(*) FROM raw_tx WHERE is_cancel = 0 AND membership_expire_date < transaction_date",
    ("transactions", "0-day plans that are not cancellations (kept, flagged)"):
        "SELECT COUNT(*) FROM raw_tx WHERE payment_plan_days = 0 AND is_cancel = 0",
    ("transactions", "Free transactions (kept, flagged)"):
        "SELECT COUNT(*) FROM raw_tx WHERE actual_amount_paid = 0",
    ("transactions", "Discounted transactions (kept, flagged)"):
        "SELECT COUNT(*) FROM raw_tx WHERE actual_amount_paid > 0 AND actual_amount_paid < plan_list_price",
    ("members", "Implausible age set to NULL"):
        "SELECT COUNT(*) FROM raw_members WHERE bd IS NULL OR bd NOT BETWEEN 10 AND 90",
    ("members", "Missing gender set to 'unknown'"):
        "SELECT COUNT(*) FROM raw_members WHERE gender IS NULL",
    ("user_logs", "Rows merged into one row per user-day"):
        "SELECT (SELECT COUNT(*) FROM raw_logs) - (SELECT COUNT(*) FROM (SELECT DISTINCT msno, date FROM raw_logs))",
    ("user_logs", "Rows with negative listening time (set to 0)"):
        "SELECT COUNT(*) FROM raw_logs WHERE total_secs < 0",
    ("user_logs", "User-days over 24 h listening (capped)"):
        """SELECT COUNT(*) FROM (SELECT msno, date, SUM(GREATEST(total_secs, 0)) AS s
                                 FROM raw_logs GROUP BY ALL) WHERE s > 86400""",
}

table_size = {
    "transactions": con.sql("SELECT COUNT(*) FROM raw_tx").fetchone()[0],
    "members":      con.sql("SELECT COUNT(*) FROM raw_members").fetchone()[0],
    "user_logs":    con.sql("SELECT COUNT(*) FROM raw_logs").fetchone()[0],
}

rows = []
for (table, rule), query in log_queries.items():
    n = con.sql(query).fetchone()[0]
    rows.append({"table": table, "rule": rule, "rows_affected": n,
                 "pct_of_table": round(100 * n / table_size[table], 2)})

cleaning_log = pd.DataFrame(rows)
cleaning_log.to_csv("reports/cleaning_log.csv", index=False)
cleaning_log

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table,rule,rows_affected,pct_of_table
0,transactions,Exact duplicate rows removed,3339,0.01
1,transactions,Payments with expiry before payment date dropped,6462,0.03
2,transactions,"0-day plans that are not cancellations (kept, ...",854136,3.72
3,transactions,"Free transactions (kept, flagged)",1218324,5.30
4,transactions,"Discounted transactions (kept, flagged)",306169,1.33
5,members,Implausible age set to NULL,4549108,67.20
6,members,Missing gender set to 'unknown',4429505,65.43
7,user_logs,Rows merged into one row per user-day,0,0.00
8,user_logs,Rows with negative listening time (set to 0),0,0.00
9,user_logs,User-days over 24 h listening (capped),4200,0.02


In [12]:
## 5. Verify the clean tables
con.sql("""
    SELECT 'tx_clean'      AS table_name, COUNT(*) AS n_rows, COUNT(DISTINCT msno) AS n_users,
           MIN(tx_date)    AS min_date,   MAX(tx_date)  AS max_date FROM tx_clean
    UNION ALL
    SELECT 'members_clean', COUNT(*), COUNT(DISTINCT msno),
           MIN(registration_date), MAX(registration_date) FROM members_clean
    UNION ALL
    SELECT 'logs_clean',    COUNT(*), COUNT(DISTINCT msno),
           MIN(log_date),  MAX(log_date) FROM logs_clean
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table_name,n_rows,n_users,min_date,max_date
0,tx_clean,22968954,2425978,2015-01-01,2017-03-31
1,members_clean,6769473,6769473,2004-03-26,2017-04-29
2,logs_clean,18396362,1103894,2017-03-01,2017-03-31


## Cleaning summary

**Transactions:** 22,978,755 → 22,968,954 rows (−9,801, 0.04%)
- Removed 3,339 exact duplicate rows (0.01%).
- Dropped 6,462 payments whose expiry date is before the payment date (0.03%). A payment should extend the membership, so these records are contradictory.
- Kept all cancellations, including 152,304 whose expiry is before the cancellation date (17% of cancellations). These plausibly reflect refunds or memberships that had already lapsed, and they show when a membership really ended.
- Kept and flagged 854,136 0-day plans (3.72%). Check 2a showed they are **not** cancellations (98% have `is_cancel = 0`); their meaning is still unclear, but their expiry dates remain usable.
- Kept and flagged 1,218,324 free transactions (5.30%) and 306,169 discounted transactions (1.33%). The discount share is lower than in Step 2 (3.8%) because free transactions are now counted separately.
- Dates converted from integers to real dates. Coverage unchanged: 2015-01-01 to 2017-03-31; 165 users lost all their transactions in cleaning.

**Members:** 6,769,473 rows, one row per user (no duplicates), nothing dropped
- Age outside 10–90 set to NULL for 4,549,108 members (67.2%), including 899 with ages 1–9. Grouped into age bands with an "Unknown" category.
- Missing gender set to "unknown" for 4,429,505 members (65.4%).
- Registration dates range from 2004-03-26 to 2017-04-29.

**User logs:** March 2017 only (`user_logs_v2`) for now
- 18,396,362 user-days for 1,103,894 users. No duplicate user-days and no negative listening times in this file.
- Capped 4,200 user-days with more than 24 hours of listening (0.02%).
- To be re-run once the full `user_logs.csv` is loaded.

**Open issues before Step 4**
- Expiry dates of 1970-01-01 (a placeholder for missing dates) and far-future expiries (up to 2036) are not yet handled. Count them and decide on a rule before building subscription spells.

**Overall:** cleaning removed less than 0.05% of transactions. Most issues were handled by flagging rather than dropping, so later steps can decide how to treat trials, free periods and 0-day plans.

In [13]:
con.close()